In [1]:
%pip install duckdb datashader colorcet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import duckdb
import datashader as ds
import datashader.transfer_functions as tf
import colorcet as cc
from PIL import Image
import os

Image.MAX_IMAGE_PIXELS = None

In [5]:
YEARS = [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
PERIODS = [f"{y}-{m:02d}" for y in YEARS for m in range(1, 13)]
SPECIES = ["*"]
OUTPUT_DIR = "/mnt/shared_data/finflow/images/monthly"
W, H = 6000, 3000

In [6]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

cvs = ds.Canvas(plot_width=W, plot_height=H, x_range=(-180, 180), y_range=(-90, 90))

for period in PERIODS:
    query_gfw = f"""
    SELECT lon, lat, hours 
    FROM read_parquet('/mnt/shared_data/finflow/gfw_raw/*/*.parquet', filename=true)
    WHERE regexp_extract(filename, '(\d{{4}}-\d{{2}})') = '{period}'
    """
    df_gfw = duckdb.query(query_gfw).to_df()
    
    img_gfw = None
    if not df_gfw.empty:
        agg_gfw = cvs.points(df_gfw, 'lon', 'lat', ds.sum('hours'))
        img_gfw = tf.shade(agg_gfw, cmap=cc.fire, how='log').to_pil().convert("RGBA")

    for spec in SPECIES:
        query_obis = f"""
        SELECT decimalLongitude as lon, decimalLatitude as lat 
        FROM read_parquet('/mnt/shared_data/finflow/obis_raw/{spec}/*/*.parquet') 
        WHERE strftime(eventDate, '%Y-%m') = '{period}'
          AND lon IS NOT NULL AND lat IS NOT NULL
        """
        df_obis = duckdb.query(query_obis).to_df()
        
        combined = Image.open("/mnt/shared_data/finflow/images/base_map.png").resize((W, H)).convert("RGBA")
        
        if img_gfw:
            combined.alpha_composite(img_gfw)
        
        if not df_obis.empty:
            agg_obis = cvs.points(df_obis, 'lon', 'lat', ds.count())
            img_obis = tf.shade(agg_obis, cmap=["#90ee90", "#00ff00"], how='log').to_pil().convert("RGBA")
            combined.alpha_composite(img_obis)
        
        file_spec = spec.replace("*", "all").replace(" ", "_")
        save_path = os.path.join(OUTPUT_DIR, f"{period}_{file_spec}.png")
        
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        
        combined.save(save_path)
        print(f"Erfolgreich gespeichert: {save_path}")

Erfolgreich gespeichert: /mnt/shared_data/finflow/images/monthly/2012-01_all.png
Erfolgreich gespeichert: /mnt/shared_data/finflow/images/monthly/2012-02_all.png
Erfolgreich gespeichert: /mnt/shared_data/finflow/images/monthly/2012-03_all.png
Erfolgreich gespeichert: /mnt/shared_data/finflow/images/monthly/2012-04_all.png
Erfolgreich gespeichert: /mnt/shared_data/finflow/images/monthly/2012-05_all.png
Erfolgreich gespeichert: /mnt/shared_data/finflow/images/monthly/2012-06_all.png
Erfolgreich gespeichert: /mnt/shared_data/finflow/images/monthly/2012-07_all.png
Erfolgreich gespeichert: /mnt/shared_data/finflow/images/monthly/2012-08_all.png
Erfolgreich gespeichert: /mnt/shared_data/finflow/images/monthly/2012-09_all.png
Erfolgreich gespeichert: /mnt/shared_data/finflow/images/monthly/2012-10_all.png
Erfolgreich gespeichert: /mnt/shared_data/finflow/images/monthly/2012-11_all.png
Erfolgreich gespeichert: /mnt/shared_data/finflow/images/monthly/2012-12_all.png
Erfolgreich gespeichert: /mn

In [10]:
import shutil
import os

SOURCE_DIR = "/mnt/shared_data/finflow/images/monthly/"
OUTPUT_FILENAME = "/mnt/shared_data/finflow/images/monthly_zip"

# Check if directory exists before zipping
if os.path.exists(SOURCE_DIR):
    print(f"Starting compression of {SOURCE_DIR}...")
    
    # make_archive automatically adds the .zip extension
    output_path = shutil.make_archive(OUTPUT_FILENAME, 'zip', SOURCE_DIR)
    
    print(f"Success! Archive created at: {output_path}")
else:
    print(f"Error: The directory {SOURCE_DIR} does not exist.")

Starting compression of /mnt/shared_data/finflow/images/monthly/...


KeyboardInterrupt: 